# Dataset 7: dynamics-change experiment

This notebook studies whether DCIts coefficients adapt when the generating dynamics change halfway through the time series.
The first half is generated with regime A, while the second half is generated with regime B.

The notebook is intended to be run from the `DCIts/examples` folder, or from an experiment folder copied inside `DCIts/examples`, so that `../src` points to the original DCIts source code.


In [ ]:
import json
import os
import pickle
import random
import sys
from contextlib import contextmanager
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn as nn

# Load DCIts utilities. This supports two layouts:
# 1. notebook inside DCIts/examples, where DCIts/src contains the utilities;
# 2. this inspection folder, where seminar-specific utilities are under
#    seminar_inspection/support_utils/src and the original DCIts clone is nearby.
def add_dcits_to_path():
    cwd = Path.cwd().resolve()
    search_bases = [cwd, *cwd.parents]

    utility_root = None
    dcits_root = None

    for base in search_bases:
        candidates = [
            base,
            base / "DCIts",
            base / "support_utils",
            base / "seminar_inspection" / "support_utils",
        ]
        for candidate in candidates:
            if utility_root is None and (candidate / "src" / "utils_DynamicsChange.py").exists():
                utility_root = candidate
            if dcits_root is None and (candidate / "src" / "dcits.py").exists():
                dcits_root = candidate

    if utility_root is None:
        raise FileNotFoundError("Could not find src/utils_DynamicsChange.py.")
    if dcits_root is None:
        raise FileNotFoundError("Could not find src/dcits.py from the original DCIts project.")

    # src is a namespace-style package here, so both roots can contribute modules.
    for candidate in [str(utility_root), str(dcits_root)]:
        if candidate not in sys.path:
            sys.path.insert(0, candidate)

    return utility_root, dcits_root

UTILITY_ROOT, DCITS_ROOT = add_dcits_to_path()
print(f"Using seminar utilities from: {UTILITY_ROOT}")
print(f"Using DCIts source from: {DCITS_ROOT}")
from src.utils_DynamicsChange import *

if torch.cuda.is_available():
    device = torch.device("cuda:0")
    print(f"Using {torch.cuda.get_device_name(device)}")
else:
    device = torch.device("cpu")
    print("CUDA is not available. Using CPU instead.")

ARTIFACT_ROOT = Path("artifacts")
RUN_NAME = "dynamics_change"
OUT_DIR = ARTIFACT_ROOT / RUN_NAME
DATA_DIR = OUT_DIR / "data"
FIG_DIR = OUT_DIR / "figures"

for path in (DATA_DIR, FIG_DIR):
    path.mkdir(parents=True, exist_ok=True)

RESULTS_PATH = DATA_DIR / "results.pkl"
STATS_PATH = DATA_DIR / "stats.pkl"
META_PATH = DATA_DIR / "meta.json"


def save_pickle(obj, path):
    with open(path, "wb") as f:
        pickle.dump(obj, f, protocol=pickle.HIGHEST_PROTOCOL)


def load_pickle(path):
    with open(path, "rb") as f:
        return pickle.load(f)


@contextmanager
def working_dir(path):
    old = Path.cwd()
    path.mkdir(parents=True, exist_ok=True)
    os.chdir(path)
    try:
        yield
    finally:
        os.chdir(old)


## Generating model

The base Dataset 7 equations are

$$
\begin{split}
X_{1,t} &= \alpha^\text{gt}_{1,1,1} X_{1,t-1} + \alpha^\text{gt}_{1,1,5} X_{1,t-5} + \epsilon_{1,t}, \\
X_{2,t} &= 1 + \alpha^\text{gt}_{2,1,2} X_{1,t-2} + \epsilon_{2,t}, \\
X_{3,t} &= \alpha^\text{gt}_{3,2,1} X_{2,t-1} + \alpha^\text{gt}_{3,4,4} X_{4,t-4} + \epsilon_{3,t}, \\
X_{4,t} &= 1 + \alpha^\text{gt}_{4,3,4} X_{3,t-4} + \alpha^\text{gt}_{4,5,1} X_{5,t-1} + \epsilon_{4,t}, \\
X_{5,t} &= \alpha^\text{gt}_{5,5,4} X_{5,t-4} + \alpha^\text{gt}_{5,2,1} X_{2,t-1} + \epsilon_{5,t}.
\end{split}
$$

The dynamics-change experiment modifies the two links into $X_3$:

- Regime A: $\alpha_{3,4,4}=1.0$, $\alpha_{3,3,4}=0.0$.
- Regime B: $\alpha_{3,4,4}=0.0$, $\alpha_{3,3,4}=0.5$.


In [ ]:
# Reproducibility
seed = 1000
random.seed(seed)
np.random.seed(seed)
torch.manual_seed(seed)

# Data-generation parameters
mean = 0.0
std = 0.1
noise_frequency = 0.3
T_total = 40000
T_half = T_total // 2

# DCIts parameters
n_runs = 5
window_length_gp = 5
temperature = 1.0
order = [1, 1]

# Dataset dimensions
no_of_timeseries_gp = 5
dataset_name = "Dataset 7"
series_names = [f"X{i}" for i in range(1, no_of_timeseries_gp + 1)]

# Ground-truth tensor: [target, source, lag_index], where lag_index 0 means lag 1.
ground_truth_alpha = torch.zeros(no_of_timeseries_gp, no_of_timeseries_gp, window_length_gp)
ground_truth_alpha[0, 0, 0] = 1 / 4
ground_truth_alpha[0, 0, 4] = 3 / 4
ground_truth_alpha[1, 0, 1] = -1
ground_truth_alpha[2, 1, 0] = 1
ground_truth_alpha[2, 3, 3] = 1
ground_truth_alpha[3, 2, 3] = -2 / 7
ground_truth_alpha[3, 4, 0] = -5 / 7
ground_truth_alpha[4, 4, 3] = 12 / 22
ground_truth_alpha[4, 1, 0] = 10 / 22

alpha_mask = (ground_truth_alpha != 0).float()
ground_truth_bias = torch.tensor([0, 1, 0, 1, 0])

# Regime-specific dynamics.
alpha_A = ground_truth_alpha.clone()
alpha_B = ground_truth_alpha.clone()

alpha_A[2, 3, 3] = 1.0
alpha_A[2, 2, 3] = 0.0

alpha_B[2, 3, 3] = 0.0
alpha_B[2, 2, 3] = 0.5


In [ ]:
print("Generating dynamics-change time series")

ts_A = dataset(
    T_half,
    alpha_A,
    ground_truth_bias,
    noise_frequency=noise_frequency,
    mu=mean,
    sigma=std,
    seed=seed,
)

# Start regime B from the last observed state of regime A.
init_state = ts_A[:, -window_length_gp:].numpy()

ts_B = dataset(
    T_half,
    alpha_B,
    ground_truth_bias,
    noise_frequency=noise_frequency,
    mu=mean,
    sigma=std,
    burn_in=0,
    seed=seed,
    init_state=init_state,
)

time_series_cd = torch.cat([ts_A, ts_B], dim=1)
print("time_series_cd shape:", tuple(time_series_cd.shape))
print("change point:", T_half)


In [ ]:
with working_dir(FIG_DIR):
    plot_ts(time_series_cd, dataset_name=dataset_name, alpha=0.7, save=True)


## Train or load DCIts results

The notebook caches `results.pkl` and `stats.pkl` under `artifacts/dynamics_change/data`.
Set `USE_CACHE = False` if you want to force a new training run.


In [ ]:
train_config = {
    "verbose": False,
    "device": device,
    "learning_rate": 1e-3,
    "scheduler_patience": 5,
    "early_stopping_modifier": 2,
    "criterion": nn.MSELoss(),
}

meta = {
    "dataset_name": dataset_name,
    "seed": seed,
    "window_length": window_length_gp,
    "n_runs": n_runs,
    "noise_frequency": noise_frequency,
    "std": std,
    "T_total": T_total,
    "T_half": T_half,
    "criterion": "MSELoss",
}
with open(META_PATH, "w", encoding="utf-8") as f:
    json.dump(meta, f, indent=2)

USE_CACHE = True

if USE_CACHE and RESULTS_PATH.exists() and STATS_PATH.exists():
    print(f"Loading cached results from {DATA_DIR}")
    results = load_pickle(RESULTS_PATH)
    stats = load_pickle(STATS_PATH)
else:
    print("No cache found. Training DCIts...")
    results = collect_multiple_runs(
        n_runs=n_runs,
        time_series=ts_A,
        different_time_series=time_series_cd,
        window_size=window_length_gp,
        temperature=temperature,
        order=order,
        config=train_config,
        seed=seed,
        verbose=True,
    )
    stats = calculate_multiple_run_statistics(results)
    save_pickle(results, RESULTS_PATH)
    save_pickle(stats, STATS_PATH)
    print(f"Saved cache to {DATA_DIR}")


In [ ]:
def plot_target_alpha_heatmap(estimated_alpha, ground_truth, target, regime_name, filename):
    """Plot one target-specific alpha heatmap using physical lag labels."""
    estimated = np.flip(np.asarray(estimated_alpha[target]), axis=1)
    truth = ground_truth[target].detach().cpu().numpy()
    vlim = max(1.0, float(np.max(np.abs(estimated))), float(np.max(np.abs(truth))))

    fig, axes = plt.subplots(1, 2, figsize=(13.5, 5.4), constrained_layout=True)
    panels = [
        (estimated, f"Estimated alpha, {regime_name}"),
        (truth, f"Ground truth alpha, {regime_name}"),
    ]

    for ax, (data, title) in zip(axes, panels):
        im = ax.imshow(data, cmap="seismic", vmin=-vlim, vmax=vlim)
        for row in range(data.shape[0]):
            for col in range(data.shape[1]):
                value = float(data[row, col])
                if abs(value) >= 0.03:
                    text_color = "white" if abs(value) > 0.45 * vlim else "black"
                    ax.text(col, row, f"{value:.2f}", ha="center", va="center", color=text_color, fontsize=11)

        ax.set_title(f"{title}\ntarget {series_names[target]}")
        ax.set_xlabel("physical lag l")
        ax.set_ylabel("source series j")
        ax.set_xticks(range(window_length_gp))
        ax.set_xticklabels(range(1, window_length_gp + 1))
        ax.set_yticks(range(no_of_timeseries_gp))
        ax.set_yticklabels(series_names)

    cbar = fig.colorbar(im, ax=axes, shrink=0.92, pad=0.02)
    cbar.set_label("alpha value")
    fig.suptitle(
        f"Dynamics-change heatmap: target {series_names[target]} (rows=sources, columns=physical lags)",
        fontsize=14,
    )
    fig.savefig(f"{filename}.png", dpi=300, bbox_inches="tight")
    fig.savefig(f"{filename}.pdf", bbox_inches="tight")
    plt.show()
    plt.close(fig)


with working_dir(FIG_DIR):
    print("Bias terms")
    print_bias(stats["alpha_bias"]["mean"], stats["alpha_bias"]["std"], ground_truth_bias)
    plot_bias(stats["alpha_bias"]["mean"])

    print("Regime A alpha coefficients")
    print_significant_alpha(stats["alpha_first"][1]["mean"], stats["alpha_first"][1]["std"], alpha_A, threshold=0.01)
    plot_target_alpha_heatmap(
        stats["alpha_first"][1]["mean"],
        alpha_A,
        target=2,
        regime_name="regime A",
        filename="regime_A_target_X3_alpha_heatmap_corrected",
    )

    print("Regime B alpha coefficients")
    print_significant_alpha(stats["alpha_second"][1]["mean"], stats["alpha_second"][1]["std"], alpha_B, threshold=0.01)
    plot_target_alpha_heatmap(
        stats["alpha_second"][1]["mean"],
        alpha_B,
        target=2,
        regime_name="regime B",
        filename="regime_B_target_X3_alpha_heatmap_corrected",
    )


## Temporal interpretation

The plots below show the mean and standard deviation of local coefficients across runs.
For the changing links, the ground-truth line is piecewise: regime A before the midpoint and regime B after the midpoint.


In [ ]:
def collect_sequence_stack(results, key):
    run_keys = [k for k in results if k.startswith("run_")]
    return np.stack([results[k][key][1] for k in run_keys], axis=0)


def flip_lag(lag, window_length):
    return window_length - lag - 1


def target_times_for_sequence(T, window_length):
    return np.arange(window_length, window_length + T)


def piecewise_truth(T, window_length, change_time, value_A, value_B):
    t_targets = target_times_for_sequence(T, window_length)
    return np.where(t_targets < change_time, abs(value_A), abs(value_B))


tracked_alphas = [
    (0, 0, 0, 1/4, 1/4),
    (0, 0, 4, 3/4, 3/4),
    (1, 0, 1, -1, -1),
    (2, 1, 0, 1, 1),
    (2, 3, 3, 1.0, 0.0),   # X4 -> X3 changes off
    (2, 2, 3, 0.0, 0.5),   # X3 -> X3 changes on
    (3, 2, 3, -2/7, -2/7),
    (3, 4, 0, -5/7, -5/7),
    (4, 4, 3, 12/22, 12/22),
    (4, 1, 0, 10/22, 10/22),
]


def plot_temporal_alphas(
    results,
    tracked_alphas,
    window_length,
    change_time,
    save_prefix="temporal_alpha",
    dpi=200,
):
    alpha_full_runs = collect_sequence_stack(results, "alpha_full")
    f_full_runs = collect_sequence_stack(results, "f_full")
    c_full_runs = collect_sequence_stack(results, "c_full")
    T = alpha_full_runs.shape[1]
    x = np.arange(T)
    change_idx = np.searchsorted(target_times_for_sequence(T, window_length), change_time)

    for target, source, lag, gt_A, gt_B in tracked_alphas:
        lag_model = flip_lag(lag, window_length)

        alpha_seq = alpha_full_runs[:, :, target, source, lag_model]
        f_seq = f_full_runs[:, :, target, source, lag_model]
        c_seq = c_full_runs[:, :, target, source, lag_model]

        mean_a, std_a = alpha_seq.mean(axis=0), alpha_seq.std(axis=0)
        mean_f, std_f = f_seq.mean(axis=0), f_seq.std(axis=0)
        mean_c, std_c = c_seq.mean(axis=0), c_seq.std(axis=0)
        gt = piecewise_truth(T, window_length, change_time, gt_A, gt_B)

        fig, axes = plt.subplots(3, 1, figsize=(11, 10), sharex=True, dpi=dpi)

        axes[0].plot(mean_f, label="mean F")
        axes[0].fill_between(x, mean_f - std_f, mean_f + std_f, alpha=0.2)
        axes[0].axvline(change_idx, color="black", linestyle="--", label="dynamics change")
        axes[0].set_ylabel("F")
        axes[0].legend()
        axes[0].grid(True, alpha=0.3)

        axes[1].plot(mean_c, label="mean C", color="tab:orange")
        axes[1].fill_between(x, mean_c - std_c, mean_c + std_c, alpha=0.2, color="tab:orange")
        axes[1].axvline(change_idx, color="black", linestyle="--", label="dynamics change")
        axes[1].set_ylabel("C")
        axes[1].legend()
        axes[1].grid(True, alpha=0.3)

        axes[2].plot(abs(mean_a), label=r"mean $|\alpha|$", color="tab:green")
        axes[2].fill_between(x, abs(mean_a - std_a), abs(mean_a + std_a), alpha=0.2, color="tab:green")
        axes[2].plot(gt, color="black", linestyle="--", linewidth=2, label="ground truth")
        axes[2].axvline(change_idx, color="black", linestyle="--", label="dynamics change")
        axes[2].set_ylabel(r"$|\alpha|$")
        axes[2].set_xlabel("window index")
        axes[2].legend()
        axes[2].grid(True, alpha=0.3)

        fig.suptitle(f"{series_names[source]} -> {series_names[target]}, lag {lag + 1}")
        fig.tight_layout(rect=[0, 0, 1, 0.96])
        fig.savefig(f"{save_prefix}_i{target+1}_j{source+1}_lag{lag+1}.png", bbox_inches="tight")
        plt.show()
        plt.close(fig)


def plot_changed_pair(results, link_a, link_b, window_length, change_time, save_name="changed_alpha_pair.png", dpi=220):
    alpha_full_runs = collect_sequence_stack(results, "alpha_full")
    T = alpha_full_runs.shape[1]
    x = np.arange(T)
    change_idx = np.searchsorted(target_times_for_sequence(T, window_length), change_time)

    fig, ax = plt.subplots(figsize=(12, 5), dpi=dpi)

    for link, color in [(link_a, "tab:blue"), (link_b, "tab:red")]:
        target, source, lag, gt_A, gt_B = link
        lag_model = flip_lag(lag, window_length)
        alpha_seq = alpha_full_runs[:, :, target, source, lag_model]
        mean_a = alpha_seq.mean(axis=0)
        std_a = alpha_seq.std(axis=0)
        gt = piecewise_truth(T, window_length, change_time, gt_A, gt_B)

        label = rf"{series_names[source]} $\to$ {series_names[target]}, lag {lag + 1}"
        ax.plot(abs(mean_a), color=color, label=label)
        ax.fill_between(x, abs(mean_a - std_a), abs(mean_a + std_a), color=color, alpha=0.2)
        ax.plot(gt, color=color, linestyle="--", linewidth=2, label=f"GT {label}")

    ax.axvline(change_idx, color="black", linestyle="--", linewidth=2, label="dynamics change")
    ax.set_xlabel("window index")
    ax.set_ylabel(r"$|\alpha|$")
    ax.set_title("Changed local coefficients")
    ax.grid(True, alpha=0.3)
    ax.legend()
    fig.tight_layout()
    fig.savefig(save_name, bbox_inches="tight")
    plt.show()
    plt.close(fig)


In [ ]:
with working_dir(FIG_DIR):
    plot_temporal_alphas(
        results,
        tracked_alphas,
        window_length=window_length_gp,
        change_time=T_half,
        save_prefix="temporal_alpha",
    )

    plot_changed_pair(
        results,
        link_a=(2, 3, 3, 1.0, 0.0),
        link_b=(2, 2, 3, 0.0, 0.5),
        window_length=window_length_gp,
        change_time=T_half,
        save_name="changed_alpha_pair_corrected.png",
    )


## Prediction quality in the second regime

The model is trained on regime A and evaluated on the full dynamics-change series.
The following cells reconstruct one-step predictions from the cached local coefficients and report RMSE on the second regime.


In [ ]:
run_key = results["summary"]["best_run"]
window_size = window_length_gp

W = create_windowed_dataset(time_series_cd, window_size)
X = W[:, :, :-1].numpy()
Y = W[:, :, -1].numpy()

t_targets = np.arange(window_size, window_size + Y.shape[0])
mask_second = t_targets >= T_half

alpha_full = results[run_key]["alpha_full"][1]
bias_full = results[run_key].get("alpha_bias_full")

# Model alpha lag axis is reversed relative to the window tensor.
X_flip = np.flip(X, axis=-1)
Yhat = np.einsum("wijl,wjl->wi", alpha_full, X_flip)

if bias_full is not None:
    b = np.diagonal(bias_full[..., 0], axis1=1, axis2=2)
    Yhat = Yhat + b

rmse_second_all = np.sqrt(np.mean((Yhat[mask_second] - Y[mask_second]) ** 2))
rmse_second_per_series = np.sqrt(np.mean((Yhat[mask_second] - Y[mask_second]) ** 2, axis=0))

print("Best run:", run_key)
print("Second-regime RMSE, all series:", rmse_second_all)
print("Second-regime RMSE per series:", rmse_second_per_series)


In [ ]:
run_keys = [k for k in results if k.startswith("run_")]
rmses = []

for run_key in run_keys:
    alpha_full = results[run_key]["alpha_full"][1]
    bias_full = results[run_key].get("alpha_bias_full")

    Yhat_run = np.einsum("wijl,wjl->wi", alpha_full, X_flip)
    if bias_full is not None:
        b = np.diagonal(bias_full[..., 0], axis1=1, axis2=2)
        Yhat_run = Yhat_run + b

    rmses.append(np.sqrt(np.mean((Yhat_run[mask_second] - Y[mask_second]) ** 2)))

rmses = np.array(rmses)
print("Second-regime RMSE across runs:", rmses.mean(), "+/-", rmses.std())


In [ ]:
def plot_second_half_predictions(Y_true, Y_pred, t_targets, change_time, n_points=15000, stride=2):
    idx = np.where(t_targets >= change_time)[0]
    idx = idx[:n_points:stride]

    for i, name in enumerate(series_names):
        fig, ax = plt.subplots(figsize=(14, 8))
        ax.plot(t_targets[idx], Y_true[idx, i], label="true", linewidth=2)
        ax.plot(t_targets[idx], Y_pred[idx, i], label="predicted", linewidth=2)
        ax.axvline(change_time, color="black", linestyle="--", linewidth=2, label="dynamics change")
        ax.set_title(f"Second-regime prediction: {name}")
        ax.set_xlabel("time")
        ax.set_ylabel("value")
        ax.grid(True, alpha=0.3)
        ax.legend()
        fig.tight_layout()
        fig.savefig(f"prediction_second_regime_{name}.png", bbox_inches="tight")
        plt.show()
        plt.close(fig)


with working_dir(FIG_DIR):
    plot_second_half_predictions(Y, Yhat, t_targets, T_half)
